# 01 – Data Storage Layer

**Projekt:** Wohnungsmarkt-Radar Wien – Preis/Leistungs-Score für Mietwohnungen

**Layer:** Data Source Layer - Identifikation der Datenquellen & Methoden der Datengewinnung

**Beschreibung:** Dieses Notebook beschreibt, wie die gescrapten Daten in MongoDB gespeichert werden. Es holt die Daten aus dem raw/runs Ordner und speichert sie in Collections. Das ganze ist in zwei Zonen aufgeteilt:

| Zone | Collections | Inhalt |
|---|---|---|
| **Rohzone** | `raw_willhaben`, `raw_immoscout24`, | Die Inserate genau so, wie die Scraper sie liefern (unterschiedliche Felder je Quelle), ergänzt um `first_seen`, `last_seen`, `status` |
| **Bereinigte Zone** | `listings_clean` | Einheitliches Schema über alle Quellen, bereinigt (fehlende Werte, Duplikate, ect), mit Lage- und Öffi-Merkmalen |


**Warum erst roh speichern und dann bereinigen?**
- Die Rohdaten bleiben erhalten. Ändern wir später eine Bereinigungsregel, müssen wir nicht nochmal scrapen.
- Die Quellen haben unterschiedliche Attribute. Ein Dokumentenspeicher nimmt das ohne festes Schema auf, das einheitliche Schema entsteht erst in der bereinigten Zone.
- `listings_clean` kann jederzeit neu aufgebaut werden.

**Zustand je Inserat:**
- `first_seen` / `last_seen`: erster und letzter Zeitpunkt, zu dem das Inserat bei einem Lauf gesehen wurde
- `status`: `active` oder `removed`. Ein Inserat gilt als entfernt, wenn es in einem **vollständigen** Lauf nicht mehr auftaucht. Schnelle Läufe (nur erste Seite) reichen dafür nicht, weil ein Inserat nur nach hinten gerutscht sein kann.
- Entfernte Inserate werden nicht gelöscht, sondern markiert. Für den Score werden nur aktive verwendet, die übrigen bleiben für weitere Auswertungen (zB. Dauer online, Verlauf der Preise) erhalten.

## 1. Imports, Konfiguration und Verbindung zu MongoDB

In [ ]:
import json
import re
import time
import warnings
from datetime import datetime, timezone
import urllib.parse
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
from pyproj import Transformer
from pymongo import MongoClient, UpdateOne
from pymongo.errors import ServerSelectionTimeoutError

# -------------------------------------------------------------------------
# 1. Pfade & externe Dateien
# -------------------------------------------------------------------------
BASE = Path.cwd().parent
RAW_DIR = BASE / "data" / "raw"
STOPS_FILE = BASE / "data" / "external" / "haltestellen.json"
BEZIRKE_FILE = BASE / "data" / "external" / "bezirksgrenzen.json"
BEZIRKE_CRS = "EPSG:31256"    # Koordinatensystem der Bezirksgrenzen (Wien)
OEFFI_RADIUS_M = 400          # Umkreis in Metern für die Anzahl der Linien

# -------------------------------------------------------------------------
# 2. Plausibilitätsgrenzen & Filter-Konstanten
# -------------------------------------------------------------------------
PRICE_RANGE = (200, 20_000)   # Miete pro Monat in Euro
AREA_RANGE = (10, 400)        # Wohnfläche in m²
VIENNA_LAT = (48.10, 48.35)   # Bounding-Box Wien Breite
VIENNA_LON = (16.17, 16.60)   # Bounding-Box Wien Länge
MIN_LISTINGS_PER_BEZIRK = 5   # Darunter wird kein Bezirks-Median berechnet

NOT_APARTMENT_RE = re.compile(r"büro|geschäftslokal|gewerbe|lager|garage|stellplatz|parkplatz", re.I)
EG_RE = re.compile(r"(?i:erdgescho(?:ss|ß)|parterre)|\bEG\b")
DG_RE = re.compile(r"(?i:dachgescho(?:ss|ß)|dachwohnung)|\bDG\b")

# Erweiterte Linientypen
LTYP_KIND = {
    1: "tram",
    2: "bus",
    3: "regionalbus",
    4: "ubahn",
    5: "sbahn",
    6: "badnerbahn"
}

# -------------------------------------------------------------------------
# 3. Verbindung zu MongoDB & Collections
# -------------------------------------------------------------------------
MONGO_URI = "mongodb://localhost:27017/"
DB_NAME = "wohnungsmarkt"  

client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=3000)

try:
    client.server_info()
    db = client[DB_NAME]
    
    print(f"Erfolgreich mit MongoDB verbunden (Datenbank: '{DB_NAME}')")
except ServerSelectionTimeoutError as e:
    print(f"Fehler: Konnte keine Verbindung zu MongoDB herstellen ({e})")

Erfolgreich mit MongoDB verbunden (Datenbank: 'wohnungsmarkt')


## 2. Rohzone - Läufe in MongoDB laden

Jede Zeile aus den runs wird als Dokument gespeichert mit dem Key/Value quelle:id als _id

- neues Inserat -> wird eingefügt, first_seen = Zeitpunkt des Laufs
- bekanntes Inserat -> Felder werden aktualisiert, last_seen wird aktualisiert
- nach einem Full-run -> aktive Inserate die nicht mehr vorkommen werden als removed markiert
- Die Läufe werden in zeitlicher Reihenfolge verabreitet. Bereits geladene Dateien merkt sich die Collection _loaded_runs -> neues Ausführen lädt also nichts doppelt

In [50]:
def read_jsonl(path):
    rows, bad = [], 0
    if not path.exists():
        print(f"{path.name} nicht gefunden – übersprungen")
        return rows
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                bad += 1
    print(f"   {path.name}: {len(rows)} Zeilen" + (f" ({bad} defekt)" if bad else ""))
    return rows


SOURCES = {
    "willhaben":   {"collection": "raw_willhaben",   "id_field": "id"},
    "immoscout24": {"collection": "raw_immoscout24", "id_field": "exposeId"},
}
FULL_RUN_MIN_SHARE = 0.5     # ein voller Lauf, der weniger als die Hälfte der bisher aktiven Inserate liefert, markiert nichts als entfernt


def sanitize(value):
    """MongoDB-Feldnamen dürfen keinen Punkt enthalten und nicht mit '$' beginnen."""
    if isinstance(value, dict):
        return {k.replace(".", "_").lstrip("$"): sanitize(v) for k, v in value.items()}
    if isinstance(value, list):
        return [sanitize(v) for v in value]
    return value


def parse_dt(text):
    return datetime.fromisoformat(text.replace("Z", "+00:00")) if text else None


def load_run_file(path, source, run_type):
    cfg = SOURCES[source]
    coll = db[cfg["collection"]]
    docs = [d for d in read_jsonl(path) if d.get(cfg["id_field"]) is not None]
    if not docs:
        return 0
    run_time = max(parse_dt(d.get("_scraped_at")) for d in docs)
    active_before = coll.count_documents({"status": "active"})

    ops = []
    for d in docs:
        seen = parse_dt(d.get("_scraped_at"))
        fields = sanitize({k: v for k, v in d.items() if k not in ("_scraped_at", "_run_type")})
        ops.append(UpdateOne(
            {"_id": f"{source}:{d[cfg['id_field']]}"},
            {"$set": {**fields, "status": "active", "last_run_type": run_type},
             "$max": {"last_seen": seen},
             "$setOnInsert": {"first_seen": seen}},
            upsert=True))
    for i in range(0, len(ops), 1000):
        coll.bulk_write(ops[i:i + 1000], ordered=False)

    # Voller Lauf: was nicht mehr auftaucht, ist entfernt (nur wenn der Lauf plausibel vollständig ist)
    if run_type == "full":
        if active_before == 0 or len(docs) >= FULL_RUN_MIN_SHARE * active_before:
            res = coll.update_many({"status": "active", "last_seen": {"$lt": run_time}},
                                   {"$set": {"status": "removed", "removed_at": run_time}})
            if res.modified_count:
                print(f"   {source}: {res.modified_count} Inserate als entfernt markiert")
        else:
            print(f"{source}: voller Lauf liefert nur {len(docs)} von {active_before} aktiven Inseraten – "
                  "nichts als entfernt markiert (Lauf unvollständig?)")
    return len(docs)


def load_all_runs():
    loaded = {d["_id"] for d in db["_loaded_runs"].find({}, {"_id": 1})}
    files = sorted((RAW_DIR / "runs").glob("*.jsonl"), key=lambda f: f.stem.split("_", 2)[2])
    new_files = [f for f in files if f.name not in loaded]
    print(f"{len(files)} Lauf-Dateien, davon {len(new_files)} noch nicht geladen")
    for f in new_files:
        source, run_type, _ = f.stem.split("_", 2)
        if source not in SOURCES:
            continue
        n = load_run_file(f, source, run_type)
        db["_loaded_runs"].insert_one({"_id": f.name, "loaded_at": datetime.now(timezone.utc), "docs": n})
        print(f"   {f.name}: {n} Inserate")
    for cfg in SOURCES.values():
        db[cfg["collection"]].create_index([("status", 1), ("last_seen", 1)])


load_all_runs()

11 Lauf-Dateien, davon 3 noch nicht geladen


**Überblick Rohzone:**

In [51]:
overview = pd.DataFrame({
    src: {"Dokumente": db[cfg["collection"]].count_documents({}),
          "aktiv": db[cfg["collection"]].count_documents({"status": "active"}),
          "entfernt": db[cfg["collection"]].count_documents({"status": "removed"})}
    for src, cfg in SOURCES.items()
})
display(overview)

example = db["raw_willhaben"].find_one({}, {"heading": 1, "price": 1,"coordinates": 1, "postcode": 1, "first_seen": 1, "last_seen": 1, "status": 1})
print("Beispieldokument (Willhaben):")
print(example)

,willhaben,immoscout24
Dokumente,3749,3001
aktiv,3739,3000
entfernt,10,1


Beispieldokument (Willhaben):
{'_id': 'willhaben:1382099182', 'coordinates': '48.2131057,16.3532645', 'first_seen': datetime.datetime(2026, 11, 9, 11, 29, 42, 711000), 'heading': 'DB IMMOBILIEN | Hochwertig sanierte Altbauwohnung in der Schlösselgasse', 'last_seen': datetime.datetime(2026, 11, 9, 11, 29, 42, 711000), 'postcode': 1080, 'price': 2100, 'status': 'removed'}


# 3. Bereinigte Zone - etablieren eines einheitlichen Schemas

Aus den zwei Rohzonen entsteht eine Tabelle mit denselben Spalten. Die Bereinigung läuft in diesen Schritten:

1. **Flags hinzufügen & Normalisieren:** Feldnamen je Quelle auf das gemeinsame Schema abbilden (Freifläche vorhanden, Privat-Angebot vorhanden, Gewerbliches-Angebot vorhanden, freie Fläche vorhanden)
3. **Filtern:** keine Büros oder Garagen, Preis und Fläche plausibel, Preis/Fläche/Zimmeranzahl vorhanden, keine Duplikate, nur Mietobjekte
4. **Merkmale:** Preis pro m², Vergleich mit dem Bezirksmedian, Öffi-Anbindung (Distanz zu den Öffis, Anzahl der Linien im Umkreis von 500 m, U-Bahn kleiner 500m Flag).

**Definition Preis:** Bei allen Quellen die monatlichen Gesamtkosten inklusive Betriebskosten.

**Hilfsfunktionen:**

In [52]:
import re

def num(x):
    try:
        return None if x in (None, "") else float(x)
    except (TypeError, ValueError):
        return None


def parse_coords(s):
    """'48.2351828,16.4810643' -> (lat, lon)"""
    try:
        lat, lon = str(s).split(",")
        return float(lat), float(lon)
    except (ValueError, AttributeError):
        return None, None

# Regex um eine 4-stellige PLZ (aus immoScout24) aus dem addressString zu extrahieren
PLZ_RE = re.compile(r'\b(1\d{3})\b')

def parse_plz(addr_string):
    if not addr_string:
        return None
    match = PLZ_RE.search(addr_string)
    return int(match.group(1)) if match else None

def safe_flag(val):
    if val is True or val == 1 or val == 1.0 or str(val).lower() in ["true", "1"]:
        return 1
    return 0

def record(source, id_, url, title, price, area, rooms, plz, lat, lon, has_outdoor, is_rent, is_buy, is_private, is_commercial, address, street=None):
    return dict(
        source=source, 
        id=str(id_), 
        url=url, 
        title=title, 
        plz=int(plz) if plz is not None else None, 
        address=address, 
        street=street,
        price=num(price), 
        area_m2=num(area), 
        rooms=num(rooms), 
        has_outdoor_space=int(has_outdoor),
        is_rent=int(is_rent),
        is_buy=int(is_buy),
        is_private=int(is_private),
        is_commercial=int(is_commercial),
        lat=lat, 
        lon=lon
    )

**1. Normalisieren & Flags setzen:**

In [53]:
def norm_willhaben(r):
    lat, lon = parse_coords(r.get("coordinates"))
    #Hilfsvariablen
    seo = r.get("seo_url")
    seo_url_lower = str(seo).lower() if seo else ""

    raw_price = r.get("price", 0)
    try:
        price_val = float(raw_price) if raw_price is not None else 0
    except (ValueError, TypeError):
        price_val = 0

    #Anbieter-Flags definieren
    is_private = safe_flag(r.get("isPrivate"))
    is_commercial = 1 - is_private
    has_rent_key = "rent/per_month_lettings" in r or (0 < price_val < 50000)
    is_rent = 1 if ("/mietwohnungen/" in seo_url_lower or has_rent_key) else 0
    is_buy = 1 - is_rent

    # Outdoor-Flag
    free_area_name = str(r.get("free_area_type_name", "")).lower()
    free_area_total = r.get("free_area/free_area_area_total", 0) or 0
    has_outdoor = 1 if (free_area_name in ["balkon", "terrasse", "garten", "loggia"] or free_area_total > 0) else 0

    return record(
        "willhaben", r.get("id"),
        f"https://www.willhaben.at/iad/{seo}" if seo else None,
        r.get("heading"),
        r.get("price"),
        r.get("estate_size") or r.get("estate_size/living_area"),
        r.get("number_of_rooms"),
        num(r.get("postcode")),
        lat, lon,
        has_outdoor,
        is_rent,
        is_buy,
        is_private,
        is_commercial,
        r.get("location"))


def norm_immoscout(r, stats):
    meta = r.get("meta") or {}
    transfer_type = str(meta.get("transferType", "")).upper()
    
    if meta.get("transferType") != "RENT":
        stats["kein_rent"] += 1
        return None
    if meta.get("hasChildren") or str(r.get("displayType", "")).startswith("DEVELOPER"):
        stats["projekt"] += 1          # Bauprojekte haben Spannen statt Einzelwerten
        return None
    
    loc = r.get("location") or {}
    addr = r.get("addressString") or ""

    # Prüfe, ob es eine Freifläche gibt
    has_outdoor = 0
    badges = r.get("typedBadges", [])
    if isinstance(badges, list):
        for b in badges:
            details = b.get("details", {}) if isinstance(b, dict) else {}
            val = str(details.get("value", "")).upper()
            if any(kw in val for kw in ["BALCONY", "TERRACE", "GARDEN", "LOGGIA"]):
                has_outdoor = 1
                break
    if not has_outdoor and any(kw in str(r).upper() for kw in ["BALCONY", "BALKON", "TERRACE", "TERRASSE"]):
        has_outdoor = 1

    # PLZ aus addressString extrahieren
    plz_val = parse_plz(addr)

    # Zur Sicherheit prüfen, ob es Kauf ist
    is_rent = 1 if transfer_type == "RENT" else 0
    is_buy = 1 if transfer_type != "RENT" else 0

    # Anbieter-Flags definieren
    is_private = safe_flag(r.get("isPrivate"))
    is_commercial = 1 - is_private

    return record(
        "immoscout24", 
        r.get("exposeId"),
        (r.get("links") or {}).get("absoluteURL"),
        r.get("headline"),
        r.get("primaryPrice"), 
        r.get("primaryArea"), 
        r.get("numberOfRooms"),
        plz_val,
        num(loc.get("lat")), 
        num(loc.get("lon")),
        has_outdoor,
        is_rent,
        is_buy,
        is_private,
        is_commercial,
        addr
    )


Beispiel:

In [54]:
import json

# Funktion zum Finden und Ausgeben des ersten validen Eintrags
def print_first_valid(source_name, collection, norm_func):
    for raw in collection.find():
        # Da norm_immoscout ein stats-Dict verlangt:
        if source_name == "immoscout24":
            stats = {"kein_rent": 0, "projekt": 0}
            res = norm_func(raw, stats)
        else:
            res = norm_func(raw)
            
        if res:  # Wenn es nicht durchgefiltert wurde
            print(f"\n--- BEISPIEL: {source_name.upper()} (Normalisiert) ---")
            print(json.dumps(res, indent=2, ensure_ascii=False))
            break

# Ausführen für beide
print_first_valid("willhaben", db["raw_willhaben"], norm_willhaben)
print_first_valid("immoscout24", db["raw_immoscout24"], norm_immoscout)


--- BEISPIEL: WILLHABEN (Normalisiert) ---
{
  "source": "willhaben",
  "id": "1382099182",
  "url": "https://www.willhaben.at/iad/immobilien/d/mietwohnungen/wien/wien-1080-josefstadt/db-immobilien-hochwertig-sanierte-altbauwohnung-in-der-schloesselgasse-1382099182/",
  "title": "DB IMMOBILIEN | Hochwertig sanierte Altbauwohnung in der Schlösselgasse",
  "plz": 1080,
  "address": "Wien, 08. Bezirk, Josefstadt",
  "street": null,
  "price": 2100.0,
  "area_m2": 114.0,
  "rooms": 4.0,
  "has_outdoor_space": 0,
  "is_rent": 1,
  "is_buy": 0,
  "is_private": 0,
  "is_commercial": 1,
  "lat": 48.2131057,
  "lon": 16.3532645
}

--- BEISPIEL: IMMOSCOUT24 (Normalisiert) ---
{
  "source": "immoscout24",
  "id": "697ca7323311816f3c39a96a",
  "url": "https://www.immobilienscout24.at/expose/697ca7323311816f3c39a96a",
  "title": "Traumwohnungen in Hietzing",
  "plz": 1130,
  "address": "1130 Wien,Hietzing",
  "street": null,
  "price": 4582.87,
  "area_m2": 144.85,
  "rooms": 3.0,
  "has_outdoor_s

**Öffi-Haltestellen einlesen & für jede Wohnung nearest ubahn, Anzahl der jeweiligen Öffis, sowie die Gesamtanzahl der Öffis im Umrkeis von 500 Metern :**

In [ ]:
# Dein neues Mapping als globale Konstante
LTYP_KIND = {
    1: "tram",
    2: "bus",
    3: "regionalbus",
    4: "ubahn",
    5: "sbahn",
    6: "badnerbahn"
}

def load_stops(path):
    """Wiener Linien OGD-Haltestellen angepasst an LTYP_KIND."""
    with open(path, encoding="utf-8") as f:
        features = json.load(f)["features"]
        
    lat, lon, kind, lines, names = [], [], [], [], []
    for ft in features:
        geom, props = ft.get("geometry") or {}, ft.get("properties") or {}
        if geom.get("type") != "Point":
            continue
            
        # Linien bereinigen (Nachtlinien ignorieren)
        tokens = {t.strip() for t in str(props.get("HLINIEN") or "").split(",") if t.strip()}
        day_lines = frozenset(t for t in tokens if not NIGHT_LINE_RE.match(t))
        if not day_lines:
            continue
            
        # Typ über LTYP_KIND ermitteln
        ltyp = props.get("LTYP")
        k = LTYP_KIND.get(ltyp)
        if not k:
            continue  # Unbekannte oder ignorierte Typen überspringen
            
        lon_, lat_ = geom["coordinates"][:2]  # GeoJSON: erst lon, dann lat!
        lat.append(lat_)
        lon.append(lon_)
        kind.append(k)
        lines.append(day_lines)
        names.append(props.get("HTXT"))
        
    stops = dict(
        lat=np.radians(lat), 
        lon=np.radians(lon), 
        kind=np.array(kind), 
        lines=lines,
        names=np.array(names, dtype=object)
    )
    
    # Counts für alle definierten Typen ausgeben
    valid_kinds = list(LTYP_KIND.values())
    counts = {k: int((stops["kind"] == k).sum()) for k in valid_kinds}
    counts_str = ", ".join([f"{k.capitalize()} {counts[k]}" for k in valid_kinds])
    print(f"   {len(kind)} Haltestellen geladen ({counts_str})")
    
    if counts["ubahn"] == 0:
        print("Keine U-Bahn-Haltestellen erkannt – bitte LTYP / HLINIEN prüfen.")
        
    return stops


def oeffi_features(lat, lon, stops, radius=OEFFI_RADIUS_M, chunk=1000):
    """Je Wohnung: Distanz zur nächsten U-Bahn (m) + Anzahl verschiedener Linien je Typ im Umkreis."""
    lat, lon = np.radians(np.asarray(lat, float)), np.radians(np.asarray(lon, float))
    n = len(lat)
    
    # Output-Dictionary um alle Verkehrsmittel aus LTYP_KIND erweitert
    out = dict(
        dist_ubahn_m=np.full(n, np.nan), 
        nearest_ubahn=np.full(n, None, dtype=object),
        n_ubahn_lines=np.zeros(n, int), 
        n_tram_lines=np.zeros(n, int),
        n_bus_lines=np.zeros(n, int), 
        n_regionalbus_lines=np.zeros(n, int),
        n_sbahn_lines=np.zeros(n, int), 
        n_badnerbahn_lines=np.zeros(n, int),
        n_lines=np.zeros(n, int)
    )
    
    kinds = stops["kind"]
    ubahn_kinds = ["ubahn"]
    u_idx = np.where(np.isin(kinds, ubahn_kinds))[0]
    
    valid_kinds = list(LTYP_KIND.values())
    
    for i in range(0, n, chunk):
        la, lo = lat[i:i + chunk, None], lon[i:i + chunk, None]
        a = np.sin((stops["lat"] - la) / 2) ** 2 + np.cos(la) * np.cos(stops["lat"]) * np.sin((stops["lon"] - lo) / 2) ** 2
        d = 2 * 6_371_000 * np.arcsin(np.sqrt(a))
        
        if len(u_idx):
            du = d[:, u_idx]
            j = du.argmin(axis=1)
            out["dist_ubahn_m"][i:i + chunk] = du[np.arange(len(j)), j]
            out["nearest_ubahn"][i:i + chunk] = stops["names"][u_idx][j]
            
        within = d <= radius
        for r in range(d.shape[0]):
            idx = np.where(within[r])[0]
            by_kind = {k: set() for k in valid_kinds}
            
            for k_idx in idx:
                k_val = kinds[k_idx]
                if k_val in by_kind:
                    by_kind[k_val] |= stops["lines"][k_idx]
                    
            row = i + r
            out["n_ubahn_lines"][row] = len(by_kind["ubahn"])
            out["n_tram_lines"][row] = len(by_kind["tram"])
            out["n_bus_lines"][row] = len(by_kind["bus"])
            out["n_regionalbus_lines"][row] = len(by_kind["regionalbus"])
            out["n_sbahn_lines"][row] = len(by_kind["sbahn"])
            out["n_badnerbahn_lines"][row] = len(by_kind["badnerbahn"])
            
            # Alle Linien über alle Typen hinweg aggregieren (mehrere Haltestellen der gleichen Linie können im Umkreis liegen)
            all_lines = set().union(*by_kind.values())
            out["n_lines"][row] = len(all_lines)
            
    out["dist_ubahn_m"] = out["dist_ubahn_m"].round(0)
    return out

Beispiel:

In [71]:
import json
import numpy as np

def print_first_valid_with_oeffi(source_name, collection, norm_func, stops):
    for raw in collection.find():
        # 1. Normalisierung wie gehabt
        if source_name == "immoscout24":
            stats = {"kein_rent": 0, "projekt": 0}
            res = norm_func(raw, stats)
        else:
            res = norm_func(raw)
            
        if res:  # Wenn das Listing gültig ist (nicht herausgefiltert)
            # 2. Koordinaten aus dem normalisierten Ergebnis holen 
            # (Angenommen res hat keys "lat" und "lon", ansonsten anpassen)
            lat_val = res.get("lat")
            lon_val = res.get("lon")
            
            if lat_val is not None and lon_val is not None:
                # 3. Öffi-Features temporär für diese eine Koordinate berechnen
                oeffi_res = oeffi_features(lat=[lat_val], lon=[lon_val], stops=stops)
                
                # 4. Öffi-Werte sauber als Unter-Dictionary (oder direkt) an das Ergebnis hängen
                # Direkt flach ins Listing-Dictionary schreiben (wie price):
                res["dist_ubahn_m"] = float(oeffi_res["dist_ubahn_m"][0])
                res["nearest_ubahn"] = str(oeffi_res["nearest_ubahn"][0])
                res["n_tram_lines"] = int(oeffi_res["n_tram_lines"][0])
                res["n_bus_lines"] = int(oeffi_res["n_bus_lines"][0])
                res["n_regionalbus_lines"] = int(oeffi_res["n_regionalbus_lines"][0])
                res["n_sbahn_lines"] = int(oeffi_res["n_sbahn_lines"][0])
                res["n_badnerbahn_lines"] = int(oeffi_res["n_badnerbahn_lines"][0])
                res["n_lines"] = int(oeffi_res["n_lines"][0])
                
            else:
                res["oeffi_features"] = "Keine gültigen Koordinaten gefunden"

            # 5. Ausgeben im gewohnten Stil
            print(f"\n--- BEISPIEL: {source_name.upper()} (Normalisiert inkl. Öffi-Features im RAM) ---")
            print(json.dumps(res, indent=2, ensure_ascii=False))
            break

# --- Anwendung ---
NIGHT_LINE_RE = re.compile(r"^N")
stops = load_stops("../data/external/haltestellen.json")
print_first_valid_with_oeffi("willhaben", db["raw_willhaben"], norm_willhaben, stops)
print_first_valid_with_oeffi("immoscout24", db["raw_immoscout24"], norm_immoscout, stops)

   7213 Haltestellen geladen (Tram 957, Bus 3558, Regionalbus 2459, Ubahn 99, Sbahn 140, Badnerbahn 0)

--- BEISPIEL: WILLHABEN (Normalisiert inkl. Öffi-Features im RAM) ---
{
  "source": "willhaben",
  "id": "1382099182",
  "url": "https://www.willhaben.at/iad/immobilien/d/mietwohnungen/wien/wien-1080-josefstadt/db-immobilien-hochwertig-sanierte-altbauwohnung-in-der-schloesselgasse-1382099182/",
  "title": "DB IMMOBILIEN | Hochwertig sanierte Altbauwohnung in der Schlösselgasse",
  "plz": 1080,
  "address": "Wien, 08. Bezirk, Josefstadt",
  "street": null,
  "price": 2100.0,
  "area_m2": 114.0,
  "rooms": 4.0,
  "has_outdoor_space": 0,
  "is_rent": 1,
  "is_buy": 0,
  "is_private": 0,
  "is_commercial": 1,
  "lat": 48.2131057,
  "lon": 16.3532645,
  "dist_ubahn_m": 376.0,
  "nearest_ubahn": "Rathaus",
  "n_tram_lines": 5,
  "n_bus_lines": 4,
  "n_regionalbus_lines": 0,
  "n_sbahn_lines": 0,
  "n_badnerbahn_lines": 0,
  "n_lines": 8
}

--- BEISPIEL: IMMOSCOUT24 (Normalisiert inkl. Öffi